# RAG GPU-Pipeline
### Ein Forschungs- und Engineering-Notebook zur GPU-beschleunigten End-to-End-RAG-Pipeline


---

## Einordnung in die Plattform

Dieses Notebook ist die GPU-Variante der fünfstufigen RAG-Pipeline aus den Notebooks
01 bis 05. Es baut **nicht** auf neuer Logik auf, sondern auf exakt denselben Komponenten
des `rag`-Pakets, also `rag.embedding`, `rag.indexing`, `rag.retrieval`, `rag.generation`
und `rag.evaluation`. Der einzige, aber folgenreiche Unterschied ist die
**Ausführungsschicht**: die rechenintensiven Stufen laufen auf der GPU.

Die GPU-Aktivierung ist in der Deployment-Architektur bereits vorgesehen:

- `Dockerfile.notebook.gpu` installiert `torch==2.7.1+cu128` (Blackwell-fähig, `sm_120`),
  `nvidia-cusparselt-cu12` und `nvidia-ml-py` für Telemetrie.
- `docker-compose.gpu.yml` reserviert NVIDIA-Geräte für `ollama`, `hub` und einen
  dedizierten `benchmark-runner`.
- `jupyterhub_config.py` reicht via `docker.types.DeviceRequest` GPUs an gespawnte
  Notebook-Container weiter, allerdings nur für das `-gpu`-Image.

Das Notebook ist so geschrieben, dass es **mit und ohne GPU** vollständig durchläuft.
Ohne CUDA fällt jede Stufe deterministisch auf die CPU zurück und benennt das explizit.
Das ist dieselbe „kontrolliert ausweichen, aber ehrlich benennen"-Linie wie in den vorherigen Notebooks
(`EMBEDDER_AVAILABLE`, `OLLAMA_AVAILABLE`).

---

### Was GPU-Beschleunigung in diesem Stack real bedeutet

Es gibt einen verbreiteten Irrtum, „RAG auf der GPU" beschleunige alles gleichmäßig.
Tatsächlich verteilt sich der Rechenaufwand sehr ungleich, und nur ein Teil profitiert
überhaupt von einer GPU:

| Stufe | GPU-Beschleunigung | Begründung |
|---|---|---|
| Embedding-Generierung | **stark** | dichte Matrixmultiplikationen, ideal für SIMT-Hardware |
| LLM-Inferenz (Ollama) | **stark** | autoregressive Transformer-Forward-Pässe |
| Vektor-Suche (FAISS) | **mittel/situativ** | erst bei großen Korpora; klein = latenzgebunden |
| Reranking (lexikalisch) | **keine** | reine String-/Mengenoperationen auf der CPU |
| Tokenisierung, I/O, Fusion | **keine** | sequenziell, speicher-/latenzgebunden |

Dieses Notebook misst diese Verteilung, statt sie zu behaupten. Der Abschnitt zur
Bottleneck-Analyse zerlegt die End-to-End-Latenz in ihre Stufen und zeigt, **wo** die
GPU wirkt und wo nicht.


---

## 0. Imports, Logging und reproduzierbarer Kontext

Wir importieren früh und setzen Logging auf `WARNING`. Für Tiefendiagnose genügt
`logging.DEBUG` auf dem jeweiligen Submodul. Alle Zufallsquellen werden geseedet,
damit Messreihen vergleichbar bleiben.

In [ ]:
import logging
import math
import time
import statistics
import gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Callable

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

for name in ["rag.embedding", "rag.indexing", "rag.retrieval", "rag.generation", "rag.evaluation"]:
    logging.getLogger(name).setLevel(logging.WARNING)
logging.basicConfig(level=logging.WARNING)

# Deterministic, comparable measurement runs.
np.random.seed(42)

# Headless-safe plotting inside the container.
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["axes.grid"] = True
matplotlib.rcParams["grid.alpha"] = 0.25
matplotlib.rcParams["font.size"] = 10

print("Imports complete. PyTorch and the rag.* layers are probed in the next sections.")

---

## 1. Einführung in GPU-beschleunigte RAG-Systeme

Ein RAG-System ist ein heterogenes Rechensystem. Es kombiniert dichte neuronale
Inferenz (Embedding, LLM) mit klassischen, eher speicher- und latenzgebundenen
Datenstrukturoperationen (invertierter Index, Fusion, Reranking). Diese Heterogenität
ist der Schlüssel zum Verständnis, **warum** eine GPU hilft und warum sie es nicht
überall tut.

### Die GPU als Durchsatzmaschine, nicht als Latenzmaschine

Eine GPU ist auf **Durchsatz** optimiert, nicht auf die Latenz einer einzelnen kleinen
Operation. Sie besitzt tausende Recheneinheiten (SMs mit vielen CUDA-Cores), die
dieselbe Instruktion auf vielen Datenelementen gleichzeitig ausführen (SIMT). Das ist
ideal für die großen Matrixmultiplikationen eines Transformer-Forward-Passes.

Daraus folgt unmittelbar die **Batch-These**: Eine GPU rentiert sich erst, wenn genug
parallele Arbeit anliegt, um ihre Recheneinheiten zu sättigen. Ein einzelner kurzer
Text durch ein Embedding-Modell nutzt vielleicht 5 % der GPU; ein Batch von 64 Texten
nutzt sie nahezu vollständig, bei kaum höherer Latenz. Genau diese Eigenschaft messen
wir später im Batch-Scaling-Abschnitt.

### Die Speicherhierarchie als eigentlicher Engpass

Der oft unterschätzte Faktor ist nicht die Rechenleistung, sondern die
**Speicherbandbreite und der Datentransfer über PCIe**. Drei Speicherebenen sind relevant:

1. **Host-RAM (CPU):** dort liegen Rohtexte, Chunks, der JSONL-Store.
2. **PCIe-Bus:** die Brücke. Jeder Tensor muss von Host zu Device kopiert werden.
   Dieser Transfer ist teuer und oft der versteckte Kostenfaktor kleiner Anfragen.
3. **VRAM (Device):** dort liegen Modellgewichte und Aktivierungen. VRAM ist knapp und
   limitiert Batch-Größe und Modellwahl.

Eine naive Pipeline, die für jeden einzelnen Text einen Host→Device-Transfer auslöst,
verschenkt den GPU-Vorteil im Transfer-Overhead. Effiziente VRAM-Nutzung und Batching
sind deshalb keine Optimierungsdetails, sondern Architekturentscheidungen.

### Präzision: fp32, fp16, bfloat16

GPUs rechnen reduzierte Genauigkeit drastisch schneller (Tensor Cores). Drei Formate:

- **fp32:** volle Genauigkeit, Referenz, höchster VRAM-Bedarf.
- **fp16:** halbe Genauigkeit, ~2× Durchsatz, ~halber VRAM, aber kleiner dynamischer
  Bereich (Overflow-Risiko). BGE-M3 ist fp16-stabil (`use_fp16=True`).
- **bfloat16:** halbe Genauigkeit, gleicher Exponentenbereich wie fp32, dadurch robuster
  gegen Overflow. EmbeddingGemma benötigt bfloat16 (`use_bfloat16=True`); fp16 würde
  bei Gemma überlaufen.

Das `rag.embedding`-Paket kodiert diese modellspezifische Wahl bereits in der Config.
`use_fp16` und `use_bfloat16` sind dort wechselseitig exklusiv. Auf der GPU wird diese
Wahl erst wirksam.


---

## 2. Architekturübersicht

Die logische Pipeline ist identisch mit den Notebooks 01 bis 05. Was sich ändert, ist die
*Platzierung* der Berechnung auf Hardware.

```text
                ┌──────────────── HOST (CPU / RAM) ─────────────────┐
   Rohdaten ──► │  Ingestion → chunks.jsonl                         │
                │  JSONL-Stores, DocumentStore, Tokenisierung,      │
                │  Fusion, Reranking, I/O                           │
                └───────────┬───────────────────────────┬──────────┘
                            │ PCIe (Host→Device)         │
                            ▼                            ▼
                ┌──────────────── DEVICE (GPU / VRAM) ──────────────┐
                │  Embedding-Forward-Pass (BGE-M3 / Gemma)          │
                │  FAISS-GPU-Suche (optional, große Korpora)        │
                │  LLM-Inferenz (Ollama, eigener Container)         │
                └───────────────────────────────────────────────────┘
```

### SOLID-orientierte Trennung bleibt erhalten

Die GPU-Variante ändert **keine** Klassenverträge. Das ist Absicht und folgt dem
Dependency-Inversion-Prinzip: `RetrievalOrchestrator` hängt vom abstrakten
`BaseDenseIndex` ab, nicht von „CPU-FAISS" oder „GPU-FAISS". Der `BaseEmbedder`-Vertrag
ist identisch, egal ob `device="cpu"` oder `device="cuda"`. Dadurch ist die GPU keine
parallele Codebasis, sondern eine Konfigurationsentscheidung, nämlich `EmbeddingConfig.device`.

### Drei reale GPU-Pfade in diesem Stack

1. **Embedding (in-Prozess):** `EmbeddingConfig(device="cuda")`. PyTorch/sentence-transformers
   bzw. FlagEmbedding laden die Gewichte ins VRAM.
2. **LLM-Inferenz (out-of-Prozess):** Ollama läuft als eigener Container mit
   GPU-Reservierung (`docker-compose.gpu.yml`). Die Beschleunigung passiert dort, nicht
   im Notebook-Kernel. Wir messen sie über die HTTP-Latenz und die `eval_count`-Token.
3. **FAISS-GPU (optional):** Das Paket-`FAISSIndex` ist bewusst CPU-basiert
   (`faiss-cpu` in `pyproject.toml`). Wir zeigen den GPU-Pfad als Erweiterung direkt über
   die FAISS-API und begründen, warum er erst bei großen Korpora lohnt.


---

## 3. GPU-Ressourcenanalyse

Bevor wir Workloads platzieren, inventarisieren wir die Hardware. Zwei unabhängige
Quellen: **PyTorch** (Sicht des Frameworks) und **NVML/pynvml** über den
`GpuMonitor` aus `rag.evaluation.monitors` (Sicht des Treibers). Der `GpuMonitor` ist
auf kontrolliertes Ausweichen ausgelegt. Fehlt pynvml oder ein Treiber, meldet er `available = False`
statt zu werfen.

In [ ]:
try:
    import torch
    TORCH_AVAILABLE = True
except Exception as exc:  # pragma: no cover
    torch = None
    TORCH_AVAILABLE = False
    print("PyTorch not importable:", exc)

GPU_AVAILABLE = bool(TORCH_AVAILABLE and torch.cuda.is_available())
DEVICE = "cuda" if GPU_AVAILABLE else "cpu"

print(f"PyTorch verfuegbar : {TORCH_AVAILABLE}")
if TORCH_AVAILABLE:
    print(f"torch.__version__  : {torch.__version__}")
    print(f"CUDA verfuegbar    : {torch.cuda.is_available()}")
    print(f"CUDA-Build         : {torch.version.cuda}")
print(f"Gewaehltes DEVICE  : {DEVICE}")
print()
if not GPU_AVAILABLE:
    print("HINWEIS: Keine CUDA-GPU sichtbar. Das Notebook laeuft vollstaendig auf CPU.")
    print("Alle 'GPU'-Abschnitte degradieren deterministisch und benennen das explizit.")

In [ ]:
from rag.evaluation.monitors import GpuMonitor

gpu_monitor = GpuMonitor(gpu_index=0)

print(f"GpuMonitor.available : {gpu_monitor.available}")
info = gpu_monitor.info()
if info.available:
    print(f"  GPU-Name         : {info.name}")
    print(f"  VRAM gesamt      : {info.memory_total_mb:,.0f} MB")
    print(f"  VRAM belegt      : {info.memory_used_mb:,.0f} MB")
    print(f"  Auslastung       : {info.utilisation_percent:.0f} %")
else:
    print(f"  Grund            : {info.error}")

# PyTorch device properties as a cross-check against the NVML view.
if GPU_AVAILABLE:
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / (1024 ** 3)
    print()
    print("PyTorch device properties:")
    print(f"  Name             : {props.name}")
    print(f"  Compute Cap.     : sm_{props.major}{props.minor}")
    print(f"  Multiprozessoren : {props.multi_processor_count}")
    print(f"  VRAM gesamt      : {total_gb:.1f} GB")

**Interpretation.** Stimmen NVML- und PyTorch-Sicht überein, ist der Treiber-Stack
konsistent. Die Compute Capability (`sm_XX`) ist sicherheitskritisch: das Deployment
verwendet `cu128`, weil ältere CUDA-Builds nur bis `sm_90` Kernel mitbringen und auf
Blackwell-Karten (`sm_120`) mit „no kernel image is available" abstürzen würden, exakt
der im `Dockerfile.notebook.gpu` dokumentierte Grund. Die VRAM-Gesamtgröße ist die harte
Obergrenze für Batch-Größe und Modellwahl in den folgenden Abschnitten.

---

## 4. CUDA-Setup und Verifikation

Zwei Dinge müssen vor jeder Messung sichergestellt sein: (1) dass Kernels überhaupt auf
der Karte ausführbar sind, und (2) dass die GPU **aufgewärmt** ist.

### Warum Warmup keine Kosmetik ist

Der erste CUDA-Kernel-Aufruf eines Prozesses ist dramatisch langsamer als alle
folgenden. Gründe: Lazy-Initialisierung des CUDA-Kontexts, JIT-Kompilierung/Caching von
Kernels, Allokation des cuDNN-/cuBLAS-Workspace, erstmaliges Laden der Bibliotheken.
Misst man die **erste** Operation als Latenz, misst man im Wesentlichen die
Initialisierung, nicht die Rechenleistung. Deshalb verwerfen alle Benchmarks in diesem
Notebook (und im `BenchmarkRunner` des Pakets) Warmup-Iterationen.

In [ ]:
def cuda_synchronize():
    """Block until all queued GPU work is finished. No-op on CPU.

    GPU kernels are launched asynchronously. Without synchronization any
    perf_counter() measurement would only time the *launch*, not the work.
    """
    if GPU_AVAILABLE:
        torch.cuda.synchronize()


def gpu_warmup(iterations: int = 3, size: int = 2048):
    """Force CUDA context init + kernel compilation before measuring."""
    if not GPU_AVAILABLE:
        print("Warmup uebersprungen (CPU-Modus).")
        return
    for _ in range(iterations):
        a = torch.randn(size, size, device="cuda")
        b = torch.randn(size, size, device="cuda")
        _ = a @ b
    cuda_synchronize()
    del a, b
    torch.cuda.empty_cache()
    print(f"GPU-Warmup abgeschlossen ({iterations} Iterationen, {size}x{size} matmul).")


if GPU_AVAILABLE:
    # Cold vs. warm: the first matmul pays the initialization cost.
    t0 = time.perf_counter()
    x = torch.randn(2048, 2048, device="cuda"); _ = x @ x; cuda_synchronize()
    cold_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    _ = x @ x; cuda_synchronize()
    warm_ms = (time.perf_counter() - t0) * 1000

    print(f"Erste (kalte) Operation : {cold_ms:8.2f} ms")
    print(f"Zweite (warme) Operation: {warm_ms:8.2f} ms")
    if warm_ms > 0:
        print(f"Cold/Warm-Faktor        : {cold_ms / warm_ms:8.1f}x")
    del x; torch.cuda.empty_cache()
else:
    print("CPU-Modus: kein Cold/Warm-Effekt zu demonstrieren.")

gpu_warmup()

**Interpretation.** Der Cold/Warm-Faktor ist auf realer GPU oft zweistellig. Genau
diese Verzerrung würde jeden „ersten Query"-Benchmark unbrauchbar machen. Die Lehre für
den Produktionsbetrieb: ein RAG-Dienst muss seine Modelle beim Start aufwärmen, sonst
zahlt der erste echte Nutzer die Initialisierungskosten.

---

### Arbeitskorpus vorbereiten

Die Pipeline-Abschnitte brauchen Chunks. Falls die Artefakte aus Notebook 01
(`chunks.jsonl`) vorhanden sind, verwenden wir sie. Andernfalls erzeugen wir einen
nachvollziehbaren, fachlich kohärenten Demo-Korpus, dieselbe Fallback-Logik wie in
Notebook 03, damit das Notebook eigenständig lauffähig bleibt.

In [ ]:
import json

chunk_store_path = Path("/home/jovyan/workspace/data/processed/chunks.jsonl")

_DEMO_SENTENCES = [
    "Retrieval-Augmented Generation kombiniert einen Retriever mit einem Sprachmodell, um Antworten in einem Korpus zu verankern.",
    "Ein Embedding-Modell projiziert Text in einen dichten Vektorraum, in dem semantische Naehe als geometrische Naehe erscheint.",
    "BGE-M3 erzeugt dichte, sparse und ColBERT-Repraesentationen in einem einzigen Forward-Pass.",
    "Der Attention-Mechanismus gewichtet relevante Tokens und ist der rechenintensive Kern eines Transformers.",
    "FAISS implementiert exakte und approximierte Nearest-Neighbor-Suche ueber grossen Vektormengen.",
    "Ein HNSW-Graph erlaubt sublineare Suche durch hierarchische Navigation kleiner Welten.",
    "BM25 ist ein probabilistisches Rankingmodell mit TF-Saettigung und Laengennormalisierung.",
    "Cosine-Similaritaet misst den Winkel zwischen zwei Vektoren und ignoriert deren Betrag.",
    "Quantisierung reduziert die Praezision von Gewichten, um VRAM zu sparen und den Durchsatz zu erhoehen.",
    "Batching saettigt die GPU, indem viele Eingaben gleichzeitig durch das Modell laufen.",
    "Der PCIe-Transfer zwischen Host und Device ist bei kleinen Anfragen oft der dominante Kostenfaktor.",
    "Reranking ordnet Kandidaten nach einem zweiten, teureren Relevanzmodell neu.",
    "Ein invertierter Index bildet Terme auf die Dokumente ab, in denen sie vorkommen.",
    "Hybrid-Retrieval fusioniert dichte und lexikalische Signale, um Bedeutung und Praezision zu verbinden.",
    "Reciprocal Rank Fusion kombiniert Ergebnislisten allein anhand der Raenge, nicht der Scores.",
    "Tokens pro Sekunde ist die Standardmetrik fuer den Durchsatz autoregressiver LLM-Inferenz.",
]


def make_demo_chunks(n: int = 120):
    out = []
    for i in range(n):
        base = _DEMO_SENTENCES[i % len(_DEMO_SENTENCES)]
        extra = _DEMO_SENTENCES[(i * 7 + 3) % len(_DEMO_SENTENCES)]
        out.append({
            "id": f"chunk-{i:04d}",
            "document_id": f"doc-{i // 4:04d}",
            "text": f"{base} {extra}",
            "metadata": {"source": f"doc_{i // 4:04d}.md", "chunk_index": i},
        })
    return out


if chunk_store_path.exists():
    with chunk_store_path.open("r", encoding="utf-8") as f:
        chunks = [json.loads(line) for line in f if line.strip()]
    corpus_source = "echte Artefakte (chunks.jsonl)"
else:
    chunks = make_demo_chunks(120)
    corpus_source = "Demo-Korpus (Fallback)"

texts = [c["text"] for c in chunks]
print(f"Korpus-Quelle : {corpus_source}")
print(f"Chunks        : {len(chunks)}")
print(f"Beispiel      : {texts[0][:90]!r}...")

---

## 5. GPU-Embedding-Pipeline

Die Embedding-Generierung ist die erste und am stärksten GPU-affine Stufe. Wir laden
BGE-M3 über die unveränderte Paket-Fabrik `create_embedder`. Der einzige Unterschied zu
Notebook 02 ist `device=DEVICE` und, auf der GPU, `use_fp16=True`.

BGE-M3 wird via FlagEmbedding geladen und nutzt CUDA automatisch, wenn es verfügbar ist;
`use_fp16` halbiert dabei VRAM-Bedarf und verdoppelt näherungsweise den Durchsatz.

In [ ]:
from rag.embedding.config import EmbeddingConfig, EmbeddingBehaviorConfig, EmbeddingProjectionConfig
from rag.embedding.factory import create_embedder
from rag.embedding.model_cache import get_default_cache

embedding_config = EmbeddingConfig(
    provider="bge-m3",
    model_name="BAAI/bge-m3",
    model_version="1.0",
    device=DEVICE,
    batch_size=32,
    # fp16 only on GPU: it doubles throughput and halves VRAM, but is unstable on CPU.
    use_fp16=GPU_AVAILABLE,
    retrieval_mode="dense",
    behavior=EmbeddingBehaviorConfig(normalize=True, mode="document"),
    projection=EmbeddingProjectionConfig(target_dim=None, method="truncate", model_aware=True),
)

EMBEDDER_AVAILABLE = False
embedder = None
try:
    t0 = time.perf_counter()
    embedder = create_embedder(embedding_config, cache=get_default_cache())
    load_s = time.perf_counter() - t0
    EMBEDDER_AVAILABLE = True
    print(f"Embedder geladen : {embedder.model_type()}  (in {load_s:.1f}s)")
    print(f"Dimension        : {embedder.dimension()}")
    print(f"Device (Config)  : {embedding_config.device}")
    print(f"fp16             : {embedding_config.use_fp16}")
except Exception as exc:
    print(f"Embedder nicht verfuegbar: {exc}")
    print("Folgeabschnitte mit Embedder werden konditionell uebersprungen.")

In [ ]:
# Single-pass embedding of the whole corpus, with timing and VRAM delta.
EMBEDDINGS = None
if EMBEDDER_AVAILABLE:
    before = gpu_monitor.info()
    cuda_synchronize()
    t0 = time.perf_counter()
    vectors = embedder.embed_documents(texts)
    cuda_synchronize()
    elapsed = time.perf_counter() - t0
    after = gpu_monitor.info()

    EMBEDDINGS = np.asarray(vectors, dtype="float32")
    print(f"Eingebettet      : {len(vectors)} Chunks")
    print(f"Vektordimension  : {EMBEDDINGS.shape[1]}")
    print(f"Gesamtzeit       : {elapsed:.2f} s")
    print(f"Durchsatz        : {len(vectors) / elapsed:.1f} Chunks/s")
    if before.available and after.available:
        print(f"VRAM vorher      : {before.memory_used_mb:,.0f} MB")
        print(f"VRAM nachher     : {after.memory_used_mb:,.0f} MB")
        print(f"VRAM-Delta       : {after.memory_used_mb - before.memory_used_mb:+,.0f} MB")
else:
    print("Embedding uebersprungen (kein Embedder).")

**Interpretation.** Der Durchsatz in Chunks/s ist die zentrale Kennzahl der
Offline-Embedding-Stufe. Das VRAM-Delta zeigt den Aktivierungsspeicher eines Batches,
nicht die Modellgröße, die bereits beim Laden allokiert wurde. Auf der GPU steigt der
Durchsatz mit der Batch-Größe stark an, bis die Recheneinheiten gesättigt sind; das
quantifizieren wir im Skalierungsabschnitt.

---

## 6. GPU-FAISS-Indexierung

Hier ist architektonische Ehrlichkeit wichtiger als ein Marketing-Versprechen.

Das `rag`-Paket verwendet `faiss-cpu` (siehe `pyproject.toml`) und `FAISSIndex` ist
**bewusst CPU-basiert**. Das ist für die typischen Korpusgrößen dieser Lehrplattform die
richtige Wahl: Bei wenigen tausend Vektoren dominiert der Host→Device-Transfer jede
GPU-Ersparnis, und exakte CPU-Suche ist bereits Mikrosekunden schnell.

Wir gehen zweigleisig vor:

1. **Index über das Paket** (`IndexingOrchestrator` + `FAISSIndex`, CPU). Das ist der
   produktive Pfad, der mit dem Retrieval-Layer zusammenarbeitet.
2. **GPU-FAISS als Erweiterung** direkt über die FAISS-API, mit `try/except`. Wenn
   `faiss-gpu` installiert ist, zeigen wir den `index_cpu_to_gpu`-Pfad und seinen
   Nutzen; sonst erklären wir, ab welcher Größenordnung er lohnt.

In [ ]:
from rag.indexing.config import IndexConfig, DenseIndexConfig, SparseIndexConfig, FAISSConfig
from rag.indexing.orchestrator import IndexingOrchestrator

logging.getLogger("rag.indexing.orchestrator").setLevel(logging.ERROR)

index_base = Path("/home/jovyan/workspace/data/index_gpu")

INDEX_RESULT = None
if EMBEDDER_AVAILABLE:
    # The orchestrator joins embeddings with chunks via chunk_id and reads the
    # dense vector from the "embedding" field. We mirror the Embedding-Layer
    # record shape exactly.
    embedding_records = [
        {
            "id": f"emb-{c['id']}",
            "chunk_id": c["id"],
            "document_id": c["document_id"],
            "embedding": EMBEDDINGS[i].tolist(),
            "embedding_type": "dense",
            "metadata": c["metadata"],
        }
        for i, c in enumerate(chunks)
    ]

    faiss_cfg = IndexConfig(
        index_dir=index_base / "faiss_flat",
        mode="dense",
        dense=DenseIndexConfig(
            backend="faiss",
            metric="cosine",
            dimension=int(EMBEDDINGS.shape[1]),
            faiss=FAISSConfig(index_type="flat"),
        ),
        sparse=SparseIndexConfig(tokenizer="simple"),
    )
    try:
        t0 = time.perf_counter()
        INDEX_RESULT = IndexingOrchestrator(faiss_cfg).build(
            embeddings=iter(embedding_records),
            chunks=iter(chunks),
            persist=True,
        )
        build_s = time.perf_counter() - t0
        print(f"FAISS-Index (CPU, paketbasiert) gebaut: {INDEX_RESULT.document_count} Dokumente in {build_s:.2f}s")
        print(f"Backend: {type(INDEX_RESULT.dense_index).__name__}")
    except ImportError as exc:
        print(f"FAISS nicht installiert: {exc}")
        print("Fallback auf BruteForce im naechsten Schritt moeglich.")
else:
    print("Indexierung uebersprungen (kein Embedder).")

In [ ]:
# GPU-FAISS as an explicit extension path — honest about availability.
def try_gpu_faiss(vectors: "np.ndarray", query: "np.ndarray", k: int = 5):
    """Demonstrate the faiss GPU path if faiss-gpu is present.

    Returns (mode, latency_ms, indices) or (reason, None, None) on fallback.
    The package ships faiss-cpu, so this typically reports why GPU is skipped.
    """
    try:
        import faiss
    except ImportError:
        return "faiss nicht installiert", None, None

    n_gpus = faiss.get_num_gpus() if hasattr(faiss, "get_num_gpus") else 0
    dim = vectors.shape[1]
    # cosine == inner product on L2-normalized vectors.
    faiss.normalize_L2(vectors)
    q = query.copy().reshape(1, -1)
    faiss.normalize_L2(q)

    cpu_index = faiss.IndexFlatIP(dim)
    cpu_index.add(vectors)

    if n_gpus > 0 and hasattr(faiss, "StandardGpuResources"):
        res = faiss.StandardGpuResources()
        gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        t0 = time.perf_counter()
        _, idx = gpu_index.search(q, k)
        ms = (time.perf_counter() - t0) * 1000
        return f"GPU-FAISS ({n_gpus} GPU(s))", ms, idx[0].tolist()

    t0 = time.perf_counter()
    _, idx = cpu_index.search(q, k)
    ms = (time.perf_counter() - t0) * 1000
    return "CPU-FAISS (faiss-gpu nicht installiert)", ms, idx[0].tolist()


if EMBEDDER_AVAILABLE and EMBEDDINGS is not None:
    mode, ms, idx = try_gpu_faiss(EMBEDDINGS.copy(), EMBEDDINGS[0].copy(), k=5)
    print(f"Suchpfad   : {mode}")
    if ms is not None:
        print(f"Suchlatenz : {ms:.3f} ms (k=5, {len(EMBEDDINGS)} Vektoren)")
        print(f"Top-Treffer: {idx}")
    print()
    print("Faustregel: GPU-FAISS lohnt erst ab ~10^5-10^6 Vektoren. Darunter dominiert")
    print("der Host->Device-Transfer, und exakte CPU-Suche ist bereits mikrosekundenschnell.")
else:
    print("GPU-FAISS-Demo uebersprungen.")

**Interpretation.** Bei den hier vorliegenden Korpusgrößen ist die FAISS-Suche
latenz- und transfergebunden, nicht rechengebunden. Eine GPU würde die reine Suchzeit
kaum verbessern, aber jedes Mal PCIe-Transferkosten hinzufügen. Genau deshalb ist die
CPU-FAISS-Wahl des Pakets bei dieser Größenordnung korrekt. Die GPU-Beschleunigung der
Pipeline kommt aus Embedding und LLM-Inferenz, nicht aus der Vektorsuche.

---

## 7. GPU-Retrieval

Retrieval zur Laufzeit ist eine Kombination aus *GPU-Query-Embedding* und
*CPU-Indexsuche*. Der `RetrievalOrchestrator` bleibt unverändert; er bekommt denselben
GPU-Embedder injiziert. Die einzige GPU-Arbeit pro Query ist der eine Forward-Pass für
den Query-Vektor. Er ist kurz, zahlt aber den Host→Device-Transfer für eine einzelne
Eingabe (der ungünstigste Fall eines kleinen Batches).

In [ ]:
from rag.retrieval.orchestrator import RetrievalOrchestrator
from rag.retrieval.config import RetrievalConfig, DenseRetrievalConfig
from rag.indexing.sparse.tokenizer import Tokenizer

logging.getLogger("rag.retrieval").setLevel(logging.ERROR)

RETRIEVAL_ORCH = None
if EMBEDDER_AVAILABLE and INDEX_RESULT is not None:
    retrieval_config = RetrievalConfig(
        mode="dense",
        top_k=5,
        dense=DenseRetrievalConfig(candidate_k=20),
    )
    RETRIEVAL_ORCH = RetrievalOrchestrator(
        document_store=INDEX_RESULT.document_store,
        config=retrieval_config,
        embedder=embedder,
        dense_index=INDEX_RESULT.dense_index,
        tokenizer=Tokenizer(SparseIndexConfig(tokenizer="simple")),
    )

    query = "Wie beschleunigt Batching die Embedding-Generierung auf der GPU?"
    cuda_synchronize()
    t0 = time.perf_counter()
    results = RETRIEVAL_ORCH.retrieve(query, k=5)
    cuda_synchronize()
    ms = (time.perf_counter() - t0) * 1000

    print(f"Query        : {query!r}")
    print(f"Retrieval    : {ms:.2f} ms (inkl. GPU-Query-Embedding)")
    print(f"Ergebnisse   : {len(results)}")
    print()
    for rank, r in enumerate(results, 1):
        print(f"  {rank}. score={r['score']:+.4f}  {r['text'][:64]!r}...")
else:
    print("Retrieval uebersprungen (Embedder oder Index fehlt).")

**Interpretation.** Die Retrieval-Latenz hier enthält den GPU-Query-Embedding-Schritt.
Auf der GPU ist dieser eine Forward-Pass paradoxerweise *nicht* der ideale GPU-Fall: ein
einzelner kurzer Text sättigt die Karte nicht und zahlt vollen Transfer-Overhead. In
Hochlast-Szenarien lohnt deshalb *Query-Batching*, also mehrere gleichzeitige Anfragen
gemeinsam einbetten. Das ist der Hebel, den der Concurrency-Benchmark des
`rag.evaluation`-Pakets adressiert.

---

## 8. GPU-LLM-Inference

Die LLM-Inferenz läuft **nicht** im Notebook-Kernel, sondern im Ollama-Container, der in
`docker-compose.gpu.yml` eine eigene GPU-Reservierung erhält. Aus Sicht des Notebooks ist
das ein HTTP-Aufruf über `rag.generation.OllamaClient`. Die GPU-Beschleunigung wird
indirekt sichtbar: an der Latenz und an **Tokens/Sekunde**, die Ollama im Response
mitliefert (`eval_count`, `eval_duration`).

`eval_count / (eval_duration / 1e9)` ergibt die Generierungsgeschwindigkeit in Token/s,
die aussagekräftigste Einzelmetrik für autoregressive Inferenz. CPU-Inferenz eines
7B-Modells liegt typisch bei 3 bis 15 Token/s, GPU-Inferenz um eine Größenordnung höher.

In [ ]:
from rag.generation import GenerationConfig, OllamaClient

gen_config = GenerationConfig(
    model_name="mistral",
    base_url="http://ollama:11434",
    temperature=0.0,
    max_tokens=256,
    timeout=180.0,
)
gen_client = OllamaClient(gen_config)

OLLAMA_AVAILABLE = False
try:
    # First call is a deliberate warmup: it loads the model into (V)RAM.
    _ = gen_client.generate("Antworte mit genau einem Wort: Bereit?")
    raw = gen_client.generate("Erklaere in zwei Saetzen, was Retrieval-Augmented Generation ist.")
    OLLAMA_AVAILABLE = True

    print("Ollama erreichbar.")
    print(f"Modell  : {raw.get('model', 'unbekannt')}")
    ec, ed = raw.get("eval_count"), raw.get("eval_duration")
    if ec and ed:
        tps = ec / (ed / 1e9)
        print(f"Tokens  : {ec} generiert")
        print(f"Tempo   : {tps:.1f} Token/s")
    print(f"Antwort : {raw['response'].strip()[:160]}")
except Exception as exc:
    print(f"Ollama nicht erreichbar: {exc}")
    print("LLM-Abschnitte werden konditionell uebersprungen.")

**Interpretation.** Der erste Aufruf lädt das Modell in den GPU-Speicher (Cold Start);
deshalb wärmen wir explizit vor. Die gemessene Token/s-Rate ist der direkteste Beleg für
GPU-Beschleunigung der Generierung. Sie hängt vom Modell, der Quantisierung und der
GPU-Bandbreite ab, nicht von unserem Code.

---

## 9. End-to-End-Pipeline

Jetzt verketten wir alle Stufen zu einer Funktion `rag_answer(query)` und messen die
Latenz pro Stufe. Diese Stage-Zerlegung ist die Grundlage der Bottleneck-Analyse: sie
zeigt, welcher Anteil der Wartezeit auf GPU-Arbeit (Embedding, Generierung) und welcher
auf CPU-Arbeit (Suche, Fusion, Prompt-Bau) entfällt.

In [ ]:
from rag.generation import ContextPreparer, PromptBuilder, SimpleRAGStrategy, STRICT_RAG_TEMPLATE
from rag.evaluation.monitors import StageTimer


def rag_answer(query: str, k: int = 5):
    """Full online RAG path with per-stage timing. Returns (answer, timings_ms)."""
    timer = StageTimer()

    timer.start("retrieval")
    results = RETRIEVAL_ORCH.retrieve(query, k=k)
    cuda_synchronize()
    timer.stop("retrieval")

    timer.start("prompt_construction")
    preparer = ContextPreparer(max_context_chars=gen_config.max_context_chars)
    builder = PromptBuilder(STRICT_RAG_TEMPLATE)
    context = preparer.prepare(results)
    _ = builder.build(query, context)
    timer.stop("prompt_construction")

    timer.start("generation")
    strategy = SimpleRAGStrategy(gen_client, builder, preparer)
    gen = strategy.generate(query, results)
    timer.stop("generation")

    st = timer.to_stage_timing()
    timings = {
        "retrieval": st.retrieval_ms,
        "prompt_construction": st.prompt_construction_ms,
        "generation": st.generation_ms,
    }
    return gen.answer, timings


if EMBEDDER_AVAILABLE and INDEX_RESULT is not None and OLLAMA_AVAILABLE:
    answer, timings = rag_answer("Warum lohnt sich GPU-FAISS erst bei grossen Korpora?")
    total = sum(v for v in timings.values() if v)
    print("Stage-Latenzen (ms):")
    for stage, ms in timings.items():
        share = (ms / total * 100) if (ms and total) else 0.0
        print(f"  {stage:<20} {ms:8.1f} ms  ({share:4.1f} %)")
    print(f"  {'GESAMT':<20} {total:8.1f} ms")
    print()
    print("Antwort:")
    print(answer.strip()[:400])
else:
    print("End-to-End uebersprungen (eine Stufe nicht verfuegbar).")

---

## 10. Batch-Inference

Die Batch-These (Abschnitt 1) wird hier erstmals quantifiziert: Wie ändert sich der
Embedding-Durchsatz mit der Batch-Größe? Auf der GPU erwarten wir einen steilen Anstieg,
bis die Recheneinheiten gesättigt sind, danach Sättigung. Auf der CPU ist der Effekt
schwach, weil keine massive Parallelität zur Verfügung steht.

Wir variieren `EmbeddingConfig.batch_size` und betten jeweils dieselbe Textmenge ein.

In [ ]:
def measure_embedding_throughput(batch_sizes, sample_texts):
    """Re-instantiate the embedder per batch_size and measure chunks/s."""
    rows = []
    for bs in batch_sizes:
        cfg = EmbeddingConfig(
            provider="bge-m3", model_name="BAAI/bge-m3", model_version="1.0",
            device=DEVICE, batch_size=bs, use_fp16=GPU_AVAILABLE,
            retrieval_mode="dense",
            behavior=EmbeddingBehaviorConfig(normalize=True, mode="document"),
        )
        emb = create_embedder(cfg, cache=get_default_cache())
        # warmup (excluded)
        _ = emb.embed_documents(sample_texts[: min(bs, len(sample_texts))])
        cuda_synchronize()
        t0 = time.perf_counter()
        _ = emb.embed_documents(sample_texts)
        cuda_synchronize()
        dt = time.perf_counter() - t0
        rows.append({"batch_size": bs, "seconds": dt, "throughput": len(sample_texts) / dt})
    return rows


BATCH_ROWS = []
if EMBEDDER_AVAILABLE:
    # Use the whole corpus so larger batches have enough work to stay busy;
    # a too-small sample makes big batches look slower than they are.
    sample = texts[: min(160, len(texts))]
    BATCH_ROWS = measure_embedding_throughput([1, 4, 8, 16, 32, 64], sample)
    print(f"{'batch_size':>10} {'Zeit (s)':>10} {'Chunks/s':>12}")
    print("-" * 34)
    for r in BATCH_ROWS:
        print(f"{r['batch_size']:>10} {r['seconds']:>10.3f} {r['throughput']:>12.1f}")
else:
    print("Batch-Inference uebersprungen (kein Embedder).")

In [ ]:
if BATCH_ROWS:
    bs = [r["batch_size"] for r in BATCH_ROWS]
    tp = [r["throughput"] for r in BATCH_ROWS]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(bs, tp, marker="o", color="#2563eb")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("Batch-Groesse (log2)")
    ax.set_ylabel("Durchsatz (Chunks/s)")
    ax.set_title(f"Embedding-Durchsatz vs. Batch-Groesse  [{DEVICE.upper()}]")
    ax.set_xticks(bs); ax.set_xticklabels(bs)
    base = tp[0] if tp[0] else 1.0
    for x, y in zip(bs, tp):
        ax.annotate(f"{y/base:.1f}x", (x, y), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=8)
    plt.tight_layout(); plt.show()
    best_i = max(range(len(tp)), key=lambda i: tp[i])
    print(f"Optimaler Durchsatz bei batch_size={bs[best_i]}: {tp[best_i]:.1f} Chunks/s")
    print(f"Speedup Optimum vs. batch=1: {tp[best_i]/tp[0]:.1f}x")
    if best_i < len(tp) - 1:
        print(f"Hinweis: ueber batch={bs[best_i]} hinaus faellt der Durchsatz wieder leicht —")
        print("bei endlichem Sample dominieren dann Fixkosten pro Batch.")
else:
    print("Kein Plot (keine Batch-Messungen).")

**Interpretation.** Auf der GPU steigt der Durchsatz mit der Batch-Größe steil an, bis
die SMs gesättigt sind und ein Durchsatz-Optimum erreicht ist. Bei einem endlichen Sample
kann der Durchsatz jenseits dieses Optimums wieder leicht sinken, weil pro Batch Fixkosten
(Kernel-Launch, Transfer, Python-Overhead) anfallen, die sich auf weniger verbleibende
Eingaben verteilen. Der Gewinn liegt also in der mittleren bis großen Batch-Größe, nicht
zwingend in der größtmöglichen. Auf der CPU ist die Kurve nahezu flach: ohne massive
Parallelität bringt Batching kaum Gewinn. Diese eine Grafik begründet, warum die
Offline-Embedding-Stufe immer batchen sollte.

---

## 11. Durchsatz-Benchmark

Wir nutzen jetzt die Benchmark-Infrastruktur des Pakets (`rag.evaluation.benchmarks`)
statt Ad-hoc-Schleifen. Sie trennt Warmup von Messung, berechnet robuste Statistiken
(Median, p95, Stdabw.) und warnt bei zu wenigen Iterationen oder hoher Varianz.

Der `RetrievalBenchmark` misst die Retrieval-Latenz (inkl. GPU-Query-Embedding); aus
Latenz und Iterationszahl lässt sich der Durchsatz in Queries/s ableiten.

In [ ]:
from rag.evaluation.benchmarks.retrieval_benchmark import RetrievalBenchmark
from rag.evaluation.benchmarks.config import BenchmarkConfig
from rag.evaluation.report import BenchmarkReport

RETRIEVAL_BENCH = None
if RETRIEVAL_ORCH is not None:
    query_pool = [
        "Was ist Retrieval-Augmented Generation?",
        "Wie funktioniert Batching auf der GPU?",
        "Warum braucht ein RAG-System einen invertierten Index?",
        "Was misst die Tokens-pro-Sekunde-Metrik?",
        "Wann lohnt sich GPU-FAISS?",
    ]
    bench_cfg = BenchmarkConfig(
        name="gpu_retrieval_latency",
        n_iterations=30,
        warmup_iterations=5,
        top_k=5,
        model_name="BAAI/bge-m3",
        metadata={"device": DEVICE},
    )
    RETRIEVAL_BENCH = RetrievalBenchmark(
        config=bench_cfg, retriever=RETRIEVAL_ORCH,
        queries=query_pool, top_k=5, corpus_size=len(chunks),
    ).run()

    print(BenchmarkReport(RETRIEVAL_BENCH).summary())
    qps = 1000.0 / RETRIEVAL_BENCH.stats.mean if RETRIEVAL_BENCH.stats.mean else 0.0
    print(f"\nAbgeleiteter Durchsatz: {qps:.1f} Queries/s (1/mean_latency)")
else:
    print("Throughput-Benchmark uebersprungen (kein Retriever).")

---

## 12. Latenzbenchmark

Mittelwerte verbergen die Schwanzlatenz. Für Nutzererlebnis und SLAs zählt das p95, nicht
der Durchschnitt. Wir visualisieren die Verteilung der gemessenen Roh-Latenzen aus dem
Benchmark als Histogramm mit Median- und p95-Markern.

In [ ]:
if RETRIEVAL_BENCH is not None and RETRIEVAL_BENCH.raw_values:
    vals = RETRIEVAL_BENCH.raw_values
    s = RETRIEVAL_BENCH.stats
    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.hist(vals, bins=min(20, max(5, len(vals)//2)), color="#60a5fa", edgecolor="white")
    ax.axvline(s.median, color="#16a34a", linestyle="--", label=f"Median {s.median:.1f} ms")
    ax.axvline(s.p95, color="#dc2626", linestyle="--", label=f"p95 {s.p95:.1f} ms")
    ax.set_xlabel("Retrieval-Latenz (ms)")
    ax.set_ylabel("Haeufigkeit")
    ax.set_title(f"Latenzverteilung Retrieval  [{DEVICE.upper()}, n={s.n}]")
    ax.legend()
    plt.tight_layout(); plt.show()

    print(f"Mean {s.mean:.1f}  Median {s.median:.1f}  p95 {s.p95:.1f}  "
          f"Min {s.min:.1f}  Max {s.max:.1f}  Std {s.std_dev:.1f}  (ms)")
    if s.mean:
        print(f"Schwanzfaktor p95/Median: {s.p95 / s.median:.2f}x")
else:
    print("Latenzbenchmark uebersprungen.")

**Interpretation.** Ein p95/Median-Faktor nahe 1 bedeutet stabile Latenz. Ein großer
Faktor deutet auf Störeinflüsse hin, etwa Garbage Collection, konkurrierende Prozesse,
gelegentliche GPU-Sync-Stalls. Der `BenchmarkReport` warnt automatisch, wenn die
Standardabweichung mehr als die Hälfte des Mittelwerts beträgt; solche Läufe sollten auf
einem ruhigeren System wiederholt werden.

---

## 13. VRAM-Monitoring

VRAM ist die knappste Ressource. Wir messen die GPU-Belegung auf zwei Ebenen, während wir
eine wachsende Anzahl Texte einbetten:

- **Treiber-Sicht (NVML / `GpuMonitor`):** der vom Treiber gemeldete belegte Speicher.
- **Allocator-Sicht (`torch.cuda.max_memory_allocated`):** der von PyTorch tatsächlich für
  Tensoren allokierte Spitzenwert pro Batch.

Diese Unterscheidung ist wichtig: PyTorch verwendet einen *Caching-Allocator*, der einmal
reservierten Speicher behält und wiederverwendet. Die NVML-Sicht bleibt deshalb oft flach
(der reservierte Block ändert sich nicht), während die Allocator-Sicht den real mit der
Batch-Größe wachsenden Aktivierungsspeicher sichtbar macht.

In [ ]:
VRAM_SAMPLES = []
if EMBEDDER_AVAILABLE and GPU_AVAILABLE:
    nvml_baseline = gpu_monitor.info().memory_used_mb if gpu_monitor.available else 0.0
    for n in [1, 8, 16, 32, 64, min(128, len(texts))]:
        batch = texts[: min(n, len(texts))]
        # Reset the allocator peak so each batch is measured independently.
        torch.cuda.reset_peak_memory_stats()
        _ = embedder.embed_documents(batch)
        cuda_synchronize()
        peak_alloc_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        nvml_used = gpu_monitor.info().memory_used_mb if gpu_monitor.available else float("nan")
        VRAM_SAMPLES.append({
            "n": len(batch),
            "peak_alloc_mb": peak_alloc_mb,   # allocator view (grows with batch)
            "nvml_used_mb": nvml_used,         # driver view (often flat: caching allocator)
        })

    print(f"{'n_texte':>8} {'Alloc-Peak (MB)':>16} {'NVML belegt (MB)':>18}")
    print("-" * 44)
    for row in VRAM_SAMPLES:
        print(f"{row['n']:>8} {row['peak_alloc_mb']:>16,.1f} {row['nvml_used_mb']:>18,.0f}")
elif not GPU_AVAILABLE:
    print("VRAM-Monitoring uebersprungen: keine GPU (CPU-Modus).")
    print("psutil-RSS waere das CPU-Aequivalent (siehe rag.evaluation.monitors.MemoryMonitor).")
else:
    print("VRAM-Monitoring uebersprungen (kein Embedder).")

In [ ]:
if VRAM_SAMPLES:
    ns = [r["n"] for r in VRAM_SAMPLES]
    peak = [r["peak_alloc_mb"] for r in VRAM_SAMPLES]
    nvml = [r["nvml_used_mb"] for r in VRAM_SAMPLES]
    total = gpu_monitor.info().memory_total_mb if gpu_monitor.available else None
    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.plot(ns, peak, marker="s", color="#7c3aed", label="Allocator-Peak (PyTorch)")
    if not all(np.isnan(nvml)):
        ax.plot(ns, nvml, marker="o", color="#64748b", label="NVML belegt (Treiber)")
    if total:
        ax.axhline(total, color="#dc2626", linestyle="--", label=f"VRAM gesamt {total:,.0f} MB")
    ax.set_xlabel("Anzahl gleichzeitig eingebetteter Texte")
    ax.set_ylabel("VRAM (MB)")
    ax.set_title("VRAM-Belegung unter wachsender Last")
    ax.legend()
    plt.tight_layout(); plt.show()
else:
    print("Kein VRAM-Plot (keine Samples).")

**Interpretation.** Die Allocator-Peak-Kurve zeigt den real mit der Batch-Größe
wachsenden Aktivierungsspeicher; die Modellgewichte sind darin nicht enthalten, sie wurden
bereits beim Laden allokiert. Die NVML-Kurve verläuft demgegenüber typischerweise nahezu
flach, nicht weil nichts passiert, sondern weil der Caching-Allocator von PyTorch den
einmal reservierten Block hält und wiederverwendet; der Treiber sieht diese interne
Wiederverwendung nicht. Der Abstand zur roten Gesamt-VRAM-Linie ist der Sicherheitspuffer.
Wer ihn aufbraucht, riskiert `CUDA out of memory`. fp16 (Abschnitt 5) verschiebt den
Aktivierungsbedarf nach unten und erlaubt größere Batches im selben VRAM-Budget.

---

## 14. Vergleich CPU vs. GPU

Der direkteste Beleg für die GPU-Wirkung: dieselben Texte, dasselbe Modell, einmal auf
CPU und einmal auf GPU eingebettet. Auf einer GPU-Maschine instanziieren wir zusätzlich
einen CPU-Embedder; auf einer reinen CPU-Maschine entfällt der Vergleich ehrlich und wir
benennen das.

In [ ]:
CMP = None
if EMBEDDER_AVAILABLE:
    sample = texts[: min(48, len(texts))]

    def time_embed(cfg):
        emb = create_embedder(cfg, cache=get_default_cache())
        _ = emb.embed_documents(sample[: min(8, len(sample))])  # warmup
        cuda_synchronize()
        t0 = time.perf_counter()
        _ = emb.embed_documents(sample)
        cuda_synchronize()
        return time.perf_counter() - t0

    cpu_cfg = EmbeddingConfig(
        provider="bge-m3", model_name="BAAI/bge-m3", model_version="1.0",
        device="cpu", batch_size=32, use_fp16=False, retrieval_mode="dense",
        behavior=EmbeddingBehaviorConfig(normalize=True, mode="document"),
    )
    cpu_s = time_embed(cpu_cfg)

    if GPU_AVAILABLE:
        gpu_cfg = EmbeddingConfig(
            provider="bge-m3", model_name="BAAI/bge-m3", model_version="1.0",
            device="cuda", batch_size=32, use_fp16=True, retrieval_mode="dense",
            behavior=EmbeddingBehaviorConfig(normalize=True, mode="document"),
        )
        gpu_s = time_embed(gpu_cfg)
        CMP = {"cpu": cpu_s, "gpu": gpu_s, "n": len(sample)}
        print(f"CPU : {cpu_s:.2f} s  ({len(sample)/cpu_s:.1f} Chunks/s)")
        print(f"GPU : {gpu_s:.2f} s  ({len(sample)/gpu_s:.1f} Chunks/s)")
        print(f"Speedup GPU/CPU: {cpu_s/gpu_s:.1f}x")
    else:
        CMP = {"cpu": cpu_s, "gpu": None, "n": len(sample)}
        print(f"CPU : {cpu_s:.2f} s  ({len(sample)/cpu_s:.1f} Chunks/s)")
        print("GPU : nicht verfuegbar -> kein Vergleich. Auf der GPU-Plattform zeigt")
        print("      dieser Abschnitt den realen Speedup (typisch 5x-30x je nach Karte).")
else:
    print("Vergleich uebersprungen (kein Embedder).")

In [ ]:
if CMP and CMP["gpu"]:
    fig, ax = plt.subplots(figsize=(6, 4))
    labels = ["CPU (fp32)", "GPU (fp16)"]
    tputs = [CMP["n"] / CMP["cpu"], CMP["n"] / CMP["gpu"]]
    bars = ax.bar(labels, tputs, color=["#94a3b8", "#2563eb"])
    ax.set_ylabel("Durchsatz (Chunks/s)")
    ax.set_title("Embedding-Durchsatz: CPU vs. GPU")
    for b, v in zip(bars, tputs):
        ax.annotate(f"{v:.1f}", (b.get_x()+b.get_width()/2, v),
                    textcoords="offset points", xytext=(0, 4), ha="center")
    plt.tight_layout(); plt.show()
else:
    print("Kein CPU/GPU-Balkendiagramm (GPU fehlt).")

---

## 15. Skalierungsanalyse

Zwei Skalierungsachsen sind relevant: **Korpusgröße** (wie wächst die Indexierungszeit?)
und **Anfragelast** (wie skaliert Retrieval mit der Korpusgröße?). Wir nutzen den
`CorpusScalingExperiment` des Pakets für die Retrieval-Achse und messen die
Embedding-Zeit über wachsende Textmengen für die Offline-Achse.

In [ ]:
SCALE_ROWS = []
if EMBEDDER_AVAILABLE:
    # Offline axis: embedding time vs. number of texts.
    for n in [10, 25, 50, min(100, len(texts)), len(texts)]:
        batch = texts[: n]
        cuda_synchronize()
        t0 = time.perf_counter()
        _ = embedder.embed_documents(batch)
        cuda_synchronize()
        dt = time.perf_counter() - t0
        SCALE_ROWS.append({"n": n, "seconds": dt, "per_item_ms": dt / n * 1000})

    print(f"{'n_chunks':>9} {'Zeit (s)':>10} {'ms/Chunk':>10}")
    print("-" * 31)
    for r in SCALE_ROWS:
        print(f"{r['n']:>9} {r['seconds']:>10.3f} {r['per_item_ms']:>10.2f}")
else:
    print("Skalierungsanalyse uebersprungen (kein Embedder).")

In [ ]:
if SCALE_ROWS:
    ns = [r["n"] for r in SCALE_ROWS]
    secs = [r["seconds"] for r in SCALE_ROWS]
    per = [r["per_item_ms"] for r in SCALE_ROWS]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(ns, secs, marker="o", color="#2563eb")
    ax1.set_xlabel("Anzahl Chunks"); ax1.set_ylabel("Gesamtzeit (s)")
    ax1.set_title("Embedding-Zeit (skaliert ~linear)")
    ax2.plot(ns, per, marker="s", color="#16a34a")
    ax2.set_xlabel("Anzahl Chunks"); ax2.set_ylabel("ms pro Chunk")
    ax2.set_title("Amortisierte Kosten pro Chunk")
    plt.tight_layout(); plt.show()
    print("Die fallende 'ms/Chunk'-Kurve zeigt Amortisation: Fixkosten (Transfer,")
    print("Kernel-Launch) verteilen sich auf mehr Eingaben -> hoehere Effizienz bei Last.")
else:
    print("Kein Skalierungs-Plot.")

**Interpretation.** Die Gesamtzeit wächst näherungsweise linear mit der Chunkzahl,
aber die *amortisierten* Kosten pro Chunk fallen, weil Fixkosten (Transfer,
Kernel-Launch, Python-Overhead) sich verteilen. Das ist die quantitative Begründung,
große Korpora in wenigen großen Batches statt vielen kleinen zu verarbeiten.

---

## 16. Bottleneck-Analyse

Jetzt führen wir die Stage-Timings aus Abschnitt 9 über mehrere Anfragen zusammen und
zeigen, wo die End-to-End-Latenz tatsächlich entsteht. Das ist die wichtigste Grafik des
Notebooks: sie widerlegt oder bestätigt empirisch, welche Stufe der Engpass ist.

In [ ]:
STAGE_AGG = None
if EMBEDDER_AVAILABLE and INDEX_RESULT is not None and OLLAMA_AVAILABLE:
    queries = [
        "Was ist Retrieval-Augmented Generation?",
        "Wie wirkt Batching auf den GPU-Durchsatz?",
        "Wann lohnt GPU-FAISS gegenueber CPU-FAISS?",
    ]
    acc = {"retrieval": [], "prompt_construction": [], "generation": []}
    for q in queries:
        _, timings = rag_answer(q)
        for stage, ms in timings.items():
            if ms is not None:
                acc[stage].append(ms)
    STAGE_AGG = {s: (statistics.mean(v) if v else 0.0) for s, v in acc.items()}

    total = sum(STAGE_AGG.values())
    print("Mittlere Stage-Latenz ueber", len(queries), "Anfragen:")
    for stage, ms in STAGE_AGG.items():
        share = ms / total * 100 if total else 0
        print(f"  {stage:<20} {ms:8.1f} ms  ({share:4.1f} %)")
else:
    print("Bottleneck-Analyse uebersprungen (vollstaendige Pipeline noetig).")

In [ ]:
if STAGE_AGG:
    stages = list(STAGE_AGG.keys())
    vals = [STAGE_AGG[s] for s in stages]
    colors = {"retrieval": "#2563eb", "prompt_construction": "#94a3b8", "generation": "#dc2626"}
    fig, ax = plt.subplots(figsize=(8, 2.4))
    left = 0.0
    for s in stages:
        ax.barh(0, STAGE_AGG[s], left=left, color=colors.get(s, "#888"),
                edgecolor="white", label=s)
        if STAGE_AGG[s] > 0:
            ax.text(left + STAGE_AGG[s]/2, 0, f"{s}\n{STAGE_AGG[s]:.0f} ms",
                    ha="center", va="center", fontsize=8, color="white")
        left += STAGE_AGG[s]
    ax.set_yticks([]); ax.set_xlabel("Latenz (ms)")
    ax.set_title("End-to-End-Latenz nach Stufe (gestapelt)")
    plt.tight_layout(); plt.show()
    print("Beobachtung: In praktisch jeder lokalen RAG-Pipeline dominiert die")
    print("LLM-Generierung die Latenz. Retrieval und Prompt-Bau sind oft <5 %.")
else:
    print("Kein Bottleneck-Plot.")

**Interpretation.** In der überwiegenden Mehrheit lokaler RAG-Setups dominiert die
**LLM-Generierung** die End-to-End-Latenz, oft mit über 90 %. Retrieval und Prompt-Bau
sind Rauschen dagegen. Das hat eine scharfe Konsequenz für Optimierungsentscheidungen:
Geld und Zeit in schnellere Vektorsuche zu stecken, während die Generierung der Engpass
ist, ist fehlinvestiert. Der Hebel liegt bei Modellwahl, Quantisierung, `max_tokens` und
GPU-Beschleunigung der Inferenz, nicht bei FAISS.

---

## 17. Optimierungsstrategien

Aus den Messungen dieses Notebooks lassen sich konkrete, priorisierte Hebel ableiten,
geordnet nach erwartetem Effekt auf die typische lokale Pipeline.

**1. LLM-Inferenz zuerst (größter Hebel).** Da die Generierung die Latenz dominiert
(Abschnitt 16): GPU für Ollama aktivieren, ein quantisiertes Modell wählen
(Q4/Q5 statt fp16-Gewichten), `max_tokens` auf das fachlich nötige Minimum begrenzen,
und Streaming für die *wahrgenommene* Latenz nutzen.

**2. Embedding batchen (Offline-Hebel).** Abschnitt 10 zeigt den Durchsatzgewinn großer
Batches. Offline-Indexierung sollte die GPU mit `batch_size` ≥ 32 sättigen; bei
Hochlast-Retrieval lohnt *Query-Batching* mehrerer gleichzeitiger Anfragen.

**3. Präzision reduzieren.** fp16 (BGE-M3) bzw. bfloat16 (Gemma) verdoppeln näherungsweise
den Durchsatz und halbieren den VRAM-Bedarf, bei vernachlässigbarem Qualitätsverlust für
Retrieval. Das verschiebt die VRAM-Grenze aus Abschnitt 13 nach unten.

**4. Warmup als Betriebspflicht.** Der Cold/Warm-Faktor aus Abschnitt 4 ist real. Ein
Dienst muss Embedder und LLM beim Start aufwärmen, damit nicht der erste Nutzer die
Initialisierung zahlt.

**5. FAISS erst bei großen Korpora auf die GPU.** Bis ~10⁵ Vektoren ist CPU-FAISS optimal.
Darüber lohnt `index_cpu_to_gpu`; bei sehr großen Korpora zusätzlich approximierte Indizes
(HNSW/IVF) statt exaktem Flat-Index. Der `FAISSConfig` des Pakets unterstützt beide.

**6. VRAM bewusst bewirtschaften.** Der `ModelCache` (LRU, prozessweit) verhindert
doppeltes Laden. `torch.cuda.empty_cache()` und gezieltes Evicten freier Modelle halten
den Sicherheitspuffer (Abschnitt 13) offen.

Eine knappe Demonstration des Präzisions-Hebels, also wie viel VRAM fp16 gegenüber fp32 spart
(analytisch, modellgewichtsbasiert):

In [ ]:
# Analytical VRAM footprint of model weights by precision.
# BGE-M3 has ~568M parameters (XLM-RoBERTa-large backbone).
params_millions = 568
for label, bytes_per in [("fp32", 4), ("fp16/bf16", 2), ("int8", 1), ("int4", 0.5)]:
    gb = params_millions * 1e6 * bytes_per / (1024 ** 3)
    print(f"  {label:<10}: ~{gb:5.2f} GB nur fuer Gewichte")
print()
print("Aktivierungsspeicher (Batch-abhaengig) kommt hinzu -> siehe gemessenes")
print("VRAM-Delta in Abschnitt 13. Praezisionsreduktion wirkt auf beide Anteile.")

---

## 18. Fazit

Dieses Notebook hat die fünfstufige RAG-Pipeline der Plattform auf GPU-Ausführung gehoben,
**ohne** einen einzigen Klassenvertrag zu ändern. GPU-Beschleunigung ist hier eine
Konfigurationsentscheidung (`device`, `use_fp16`/`use_bfloat16`), kein Codefork. Das ist
der Verdienst der SOLID-orientierten Architektur aus den Notebooks 01 bis 05.

**Was empirisch belegt wurde:**

- Der **Cold/Warm-Effekt** ist real und macht Warmup zur Messvoraussetzung und
  Betriebspflicht (Abschnitt 4).
- **Embedding** ist stark GPU-affin; **Batching** ist der entscheidende Durchsatzhebel,
  mit klarer Sättigung der Karte (Abschnitte 5, 10, 15).
- **FAISS** ist bei diesen Korpusgrößen bewusst CPU-basiert; GPU-Suche lohnt erst bei
  großen Vektormengen (Abschnitt 6).
- **VRAM** ist die knappste Ressource; fp16 und bewusste Cache-Verwaltung halten den
  Sicherheitspuffer offen (Abschnitte 13, 17).
- Die **LLM-Generierung dominiert** die End-to-End-Latenz, der wichtigste Befund für
  jede Optimierungspriorisierung (Abschnitte 9, 16).

**Methodische Linie.** Jede GPU-Behauptung wurde gemessen, nicht angenommen; jede Stufe
degradiert ehrlich auf die CPU; jede Grafik ist interpretiert. Damit reiht sich das
Notebook in die Linie der vorangegangenen ein: nachvollziehbar, reproduzierbar,
architektonisch konsistent.

```text
                          GPU-Wirkung in dieser Pipeline (Zusammenfassung)
  Embedding   ████████████████████  stark, batch-skalierend
  LLM-Inferenz████████████████████  stark, dominiert die Latenz
  FAISS-Suche ████                   situativ (große Korpora)
  Reranking   ·                      keine (CPU, lexikalisch)
  Fusion/I/O  ·                      keine (sequenziell)
```
